# F_tk_apr Univariate Distribution Audit

This diagnostic isolates the `F_tk_apr` series from the latest Gold-layer `msm_timeseries.csv` and visualizes its standalone distribution for PM review.

- Histogram: high-resolution density view
- Boxplot: outlier geometry check
- Display convention: convert decimal APR to `%` only for chart labels/axes (`value * 100`).

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid")

repo_root = Path.cwd()
reports_dir = repo_root / "reports" / "msm_funding_v0"
latest_csv = max(reports_dir.rglob("msm_timeseries.csv"), key=lambda p: p.stat().st_mtime)

print(f"Using latest Gold CSV: {latest_csv}")

df = pd.read_csv(latest_csv)
if "F_tk_apr" not in df.columns:
    raise ValueError("Expected 'F_tk_apr' column in msm_timeseries.csv")

series_dec = pd.to_numeric(df["F_tk_apr"], errors="coerce").dropna()
series_pct = series_dec * 100.0

print(f"Rows with valid F_tk_apr: {len(series_pct)}")
print(
    "F_tk_apr range (%): "
    f"{series_pct.min():.4f} to {series_pct.max():.4f}"
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12), dpi=180, constrained_layout=True)

# High-resolution histogram
axes[0].hist(
    series_pct,
    bins=80,
    color="steelblue",
    edgecolor="white",
    linewidth=0.4,
    alpha=0.9,
)
axes[0].set_title("F_tk_apr Distribution (Histogram)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("F_tk_apr (%)")
axes[0].set_ylabel("Frequency")

# Boxplot for outlier geometry
sns.boxplot(x=series_pct, ax=axes[1], color="darkorange", width=0.35)
axes[1].set_title("F_tk_apr Distribution (Boxplot)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("F_tk_apr (%)")

# Keep axis labels as human-readable percentages
for ax in axes:
    ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f%%"))

plt.suptitle("Macro Sensor Univariate Audit: F_tk_apr", fontsize=16, fontweight="bold")

output_path = repo_root / "notebooks" / "f_tk_apr_univariate_audit.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
print(f"Saved diagnostic chart: {output_path}")

plt.show()